In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch10-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "conformal-prediction.py"

# 用校正集決定 ESOL 預測區間的半徑

讀入實測溶解度，按結構合併重複紀錄，隨機分成三組。用半徑二的 Morgan 指紋訓練隨機森林，再分別報告區間寬度與涵蓋筆數。 首次執行需連網下載 MoleculeNet 資料，可以用環境變數 `DEEPCHEM_DATA_DIR` 指定快取目錄。

下一格寫出可修改的程式，末格執行並顯示圖片。做練習時，把自己的程式接在繪圖程式底部，重新執行這兩格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
cache = Path(os.environ.setdefault("DEEPCHEM_DATA_DIR", str(Path(tempfile.gettempdir()) / "cheminfo-molnet")))
cache.mkdir(parents=True, exist_ok=True)
import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor

SEED = 42
figures = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figures.mkdir(parents=True, exist_ok=True)
from math import ceil
from sklearn.model_selection import train_test_split

tasks, (dataset,), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    data_dir=str(cache), save_dir=str(cache), reload=True)
# 用含立體化學的標準 SMILES 辨認同一結構，以中位數合併重複紀錄。
table = pd.DataFrame({"smiles": dataset.X, "logS": dataset.y[:, 0]})
labels = table.groupby("smiles", sort=True)["logS"].median()
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
X = np.stack([gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)) for s in labels.index])
y = labels.to_numpy()
fit, held = train_test_split(np.arange(len(y)), test_size=0.40, random_state=SEED)
cal, test = train_test_split(held, test_size=0.5, random_state=SEED)
model = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=1)
model.fit(X[fit], y[fit])
scores = np.abs(y[cal] - model.predict(X[cal]))

def radius(scores, alpha):
    k = ceil((len(scores) + 1) * (1 - alpha))
    return np.sort(scores)[k - 1] if k <= len(scores) else np.inf

q = radius(scores, 0.10)
pred = model.predict(X[test])
covered = np.abs(y[test] - pred) <= q
print(f"rows={len(table)}, fit={len(fit)}, cal={len(cal)}, test={len(test)}")
print(f"radius={q:.3f}, width={2*q:.3f}, covered={covered.sum()}/{len(test)} ({covered.mean():.1%})")
order = np.argsort(pred)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.fill_between(range(len(test)), pred[order]-q, pred[order]+q, color="#c9e3ea", label="90% prediction interval")
ax.plot(pred[order], color="#155369", label="Predicted")
for included, color, marker, label in [(True, "#287a83", ".", "Measured, inside"), (False, "#ae4928", "x", "Measured, outside")]:
    mask = covered[order] == included
    ax.scatter(np.flatnonzero(mask), y[test][order][mask], color=color, marker=marker, s=18, label=label)
ax.set(xlabel="Test molecules ordered by predicted logS", ylabel="logS (S in mol/L)")
ax.legend(ncols=2, fontsize=8)
fig.tight_layout()
fig.savefig(figures / "10-06-conformal-prediction.png", dpi=150)

In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "10-06-conformal-prediction.png")))